In [ ]:
# Phase 1 — Station Audit and Data Quality Scoring

## Step 1.1 — Namibian Station Identification

The guide originally listed NOAA ISD station IDs. These were found to be 
incompatible with the NCEI daily-summaries API. Correct GHCND station IDs 
were obtained directly from the NOAA master stations list at:
https://www.ncei.noaa.gov/pub/data/ghcn/daily/ghcnd-stations.txt

| Station Name        | Original ISD ID  | Confirmed GHCND ID  | Records  |
|---------------------|------------------|---------------------|----------|
| Windhoek (main/GSN) | 68110099999      | WA007401540         | 19,793   |
| Windhoek (alt A)    | 68110099999      | WA007400630         | 5,084    |
| Windhoek (alt B)    | 68110099999      | WA007401240         | 5,539    |
| Windhoek (alt C)    | 68110099999      | WA007401850         | 5,175    |
| Walvis Bay Airport  | 68066099999      | WAM00068098         | 7,429    |
| Gobabis             | 68108099999      | WA007878380         | 12,242   |
| Keetmanshoop        | 68104099999      | WA004192150         | 5,236    |
| Ondangwa            | 68006099999      | WAM00068006         | 6,642    |
| Rundu               | 68004099999      | WA012084750         | 17,121   |
| Mariental           | 68106099999      | WA005688170         | 5,769    |
| Lüderitz            | 68096099999      | **NOT IN GHCND**    | —        |

**Note:** Lüderitz is absent from the GHCND database entirely. 
This is a known data gap and will be flagged in the station audit memo.
Primary station for EVT analysis: WA007401540 (Windhoek GSN, 1970–2024).

In [11]:
import requests

# Download the official GHCND stations list and search for Namibia
print("Downloading GHCND stations list...")
r = requests.get('https://www.ncei.noaa.gov/pub/data/ghcn/daily/ghcnd-stations.txt')

lines = r.text.split('\n')

# Namibia stations start with 'SWA' (South West Africa — historical name)
# or 'WA' in the FIPS system
namibia_lines = [l for l in lines if l.startswith('SWA') or 
                 'WINDHOEK' in l.upper() or 
                 'WALVIS' in l.upper() or
                 'NAMIBIA' in l.upper() or
                 'GOBABIS' in l.upper() or
                 'KEETMANSHOOP' in l.upper() or
                 'ONDANGWA' in l.upper() or
                 'LUDERITZ' in l.upper() or
                 'RUNDU' in l.upper() or
                 'MARIENTAL' in l.upper()]

print(f"\nFound {len(namibia_lines)} Namibia stations:\n")
for l in namibia_lines:
    print(l)


Found 13 Namibia stations:

ASN00062004 -32.6000  149.6000  498.3    BURRUNDULLA                                 
BR047348180 -12.0500  -42.8000  490.0    BRUNDUE                                     
IN020030800  11.2800   77.5800  294.0    PERUNDURAI                                  
WA004192150 -26.5800   18.1300  980.0    KEETMANSHOOP                                
WA005688170 -24.6200   17.9700 1110.0    MARIENTAL                                   
WA007400630 -22.5500   17.0500 1640.0    WINDHOEK                                    
WA007401240 -22.5700   17.0800 1660.0    WINDHOEK                                    
WA007401540 -22.5670   17.1000 1700.0    WINDHOEK                       GSN     68110
WA007401850 -22.5800   17.1200 1740.0    WINDHOEK                                    
WA007878380 -22.5000   18.9670 1400.0    GOBABIS                                68116
WA012084750 -17.9170   19.7670 1100.0    RUNDU                                  68018
WAM00068006 -17.9330   15

In [9]:
# Diagnostic — test one station with full error detail
test_station = 'WA007401540'

params = {
    'dataset'   : 'daily-summaries',
    'stations'  : test_station,
    'startDate' : '2020-01-01',
    'endDate'   : '2020-12-31',
    'dataTypes' : 'PRCP,TMAX,TMIN',
    'format'    : 'json',
    'units'     : 'metric',
}

resp = requests.get(BASE, params=params, headers=headers)
print(f"Status: {resp.status_code}")
print(f"URL called: {resp.url}")
print(f"Response: {resp.text[:500]}")

Status: 200
URL called: https://www.ncei.noaa.gov/access/services/data/v1?dataset=daily-summaries&stations=WA007401540&startDate=2020-01-01&endDate=2020-12-31&dataTypes=PRCP%2CTMAX%2CTMIN&format=json&units=metric
Response: [
{"DATE":"2020-01-01","STATION":"WA007401540","TMAX":"34.7"}
,{"DATE":"2020-01-02","STATION":"WA007401540","TMAX":"34.0","TMIN":"17.5","PRCP":"1.0"}
,{"DATE":"2020-01-03","STATION":"WA007401540","TMAX":"32.8","TMIN":"17.7","PRCP":"0.0"}
,{"DATE":"2020-01-04","STATION":"WA007401540","TMAX":"32.8","TMIN":"18.6","PRCP":"10.9"}
,{"DATE":"2020-01-05","STATION":"WA007401540","TMAX":"32.2","TMIN":"17.8","PRCP":"0.0"}
,{"DATE":"2020-01-06","STATION":"WA007401540","TMAX":"32.2","TMIN":"17.0","PRCP":"0.5


In [10]:
import time

# Correct station IDs — NO 'GHCND:' prefix for this API endpoint
STATIONS = {
    'windhoek_main' : 'WA007401540',  # GSN station — best record
    'windhoek_a'    : 'WA007400630',
    'windhoek_b'    : 'WA007401240',
    'windhoek_c'    : 'WA007401850',
    'gobabis'       : 'WA007878380',
    'keetmanshoop'  : 'WA004192150',
    'mariental'     : 'WA005688170',
    'rundu'         : 'WA012084750',
    'ondangwa'      : 'WAM00068006',
    'walvis_bay'    : 'WAM00068098',
}

DATE_CHUNKS = [
    ('1970-01-01', '1979-12-31'),
    ('1980-01-01', '1989-12-31'),
    ('1990-01-01', '1999-12-31'),
    ('2000-01-01', '2009-12-31'),
    ('2010-01-01', '2019-12-31'),
    ('2020-01-01', '2024-12-31'),
]

for name, station_id in STATIONS.items():
    print(f"\nFetching {name} ({station_id})...")
    all_chunks = []

    for start, end in DATE_CHUNKS:
        params = {
            'dataset'   : 'daily-summaries',
            'stations'  : station_id,
            'startDate' : start,
            'endDate'   : end,
            'dataTypes' : 'PRCP,TMAX,TMIN,AWND',
            'format'    : 'json',
            'units'     : 'metric',
        }

        resp = requests.get(BASE, params=params, headers=headers)

        if resp.status_code == 200:
            try:
                data = resp.json()
                if len(data) > 0:
                    all_chunks.append(pd.DataFrame(data))
                    print(f"  ✓ {start[:4]}s: {len(data)} records")
                else:
                    print(f"  ⚠ {start[:4]}s: no data")
            except Exception as e:
                print(f"  ✗ {start[:4]}s: error — {e}")
        else:
            print(f"  ✗ {start[:4]}s: status {resp.status_code}")

        time.sleep(1)

    if all_chunks:
        df_full = pd.concat(all_chunks, ignore_index=True)
        df_full['DATE'] = pd.to_datetime(df_full['DATE'])
        df_full.to_csv(f'../data/raw/{name}_daily.csv', index=False)
        print(f"  → TOTAL: {len(df_full)} records saved to data/raw/{name}_daily.csv")
    else:
        print(f"  → No data retrieved for {name}")

print("\nAll stations done!")


Fetching windhoek_main (WA007401540)...
  ✓ 1970s: 3652 records
  ✓ 1980s: 3653 records
  ✓ 1990s: 3652 records
  ✓ 2000s: 3559 records
  ✓ 2010s: 3515 records
  ✓ 2020s: 1762 records
  → TOTAL: 19793 records saved to data/raw/windhoek_main_daily.csv

Fetching windhoek_a (WA007400630)...
  ✓ 1970s: 2831 records
  ✓ 1980s: 2253 records
  ⚠ 1990s: no data
  ⚠ 2000s: no data
  ⚠ 2010s: no data
  ⚠ 2020s: no data
  → TOTAL: 5084 records saved to data/raw/windhoek_a_daily.csv

Fetching windhoek_b (WA007401240)...
  ✓ 1970s: 3286 records
  ✓ 1980s: 2253 records
  ⚠ 1990s: no data
  ⚠ 2000s: no data
  ⚠ 2010s: no data
  ⚠ 2020s: no data
  → TOTAL: 5539 records saved to data/raw/windhoek_b_daily.csv

Fetching windhoek_c (WA007401850)...
  ✓ 1970s: 2922 records
  ✓ 1980s: 2253 records
  ⚠ 1990s: no data
  ⚠ 2000s: no data
  ⚠ 2010s: no data
  ⚠ 2020s: no data
  → TOTAL: 5175 records saved to data/raw/windhoek_c_daily.csv

Fetching gobabis (WA007878380)...
  ✓ 1970s: 3515 records
  ✓ 1980s: 220